# Avance 2. Ingeniería de características para imágenes DESI

**Proyecto:** Inferencia de propiedades físicas de galaxias a partir de imágenes multibanda.
**Instituto Tecnológico y de Estudios Superiores de Monterrey** · Maestría en Inteligencia Artificial Aplicada · Equipo 24.
**Colaboración científica:** Instituto de Astronomía, UNAM.

José Florencio Maguey Peralta — A01796727
Erick Alberto Bustos Cruz — A01378966
Luz Copelia Minutti Pérez — A01796921

**Asesores:** Dr. Iván Reyes Amezcua; Dr. Héctor Hernández Toledo; Dr. José Antonio Vázquez Mata.

---

**Fase de CRISP-ML(Q):** 2, *Data Engineering (Data Preparation)* (Studer et al. 2021).
**Objetivos del avance:** 2.3, crear nuevas características que mejoren el rendimiento de los modelos; 2.4, mitigar el riesgo de características sesgadas y acelerar la convergencia de algunos algoritmos.

## 1. Qué hacemos y por qué

Como punto de partida tenemos una red convolucional (la cadena de Katachi; Alfonzo et al. 2024) que entrenamos con las imágenes DESI y que estima la masa estelar, la tasa de formación estelar (SFR) y el índice D4000 directamente a partir de los píxeles. En las galaxias de validación obtuvo un RMSE de **0.224 dex** en masa, **0.631 dex** en SFR y **0.075** en D4000.

Esa red recibe 270,000 valores por galaxia (3 bandas × 300 × 300 píxeles) y no incorpora conocimiento físico. En este avance tomamos el camino contrario: reducimos cada imagen a **unas decenas de variables con significado físico**, medidas con los métodos habituales de la astronomía extragaláctica, y las preparamos para el aprendizaje automático:

1. **Construcción** (sección 5): fotometría, colores, gradientes de color, tamaños, concentración, asimetría, distribución de la luz y variables que dependen de la distancia.
2. **Codificación y discretización** (sección 7).
3. **Transformación y escalamiento** (secciones 8 y 9), con una prueba de convergencia.
4. **Sesgos** (sección 10): diferencias entre telescopios y con la distancia.
5. **Selección y extracción** (secciones 11 y 12): filtros, PCA, análisis factorial y eigengalaxias.
6. **Comparación** (sección 13): cuánto predicen las características frente a la red convolucional.

La pregunta que buscamos responder es:

> **¿Cuánto de la masa estelar, la SFR y el D4000 se puede predecir con unas decenas de características bien construidas, comparado con una red que usa la imagen completa?**

**Criterios generales.** Todos los ajustes (transformaciones, escaladores, selección, PCA y análisis factorial) se hacen **solo con las galaxias de entrenamiento** y se aplican sin cambios a las de validación. Las **galaxias de prueba no se usan**: las reservamos para la evaluación final del proyecto.

### Cómo ejecutar

1. Abrir el cuaderno en Colab. Funciona con CPU; si hay GPU, también se ejecuta la sección opcional 12.4 (Zoobot).
2. Ejecutar todas las celdas en orden (*Entorno de ejecución → Ejecutar todo*). La primera ejecución tarda entre 30 y 45 minutos, principalmente por la medición de las 9,303 galaxias. Si Colab se desconecta, basta con ejecutar todo de nuevo: la medición continúa desde el último bloque guardado.
3. El cuaderno **solo lee** el catálogo, los FITS y la caché de imágenes. Los resultados se guardan en `Mi unidad/resultados_fe/`.

## 2. Parámetros

Las constantes de la medición (umbrales de segmentación, definición del radio de Petrosian, coeficientes de extinción y cosmología) están en `src/caracteristicas_desi/config.py`, cada una con su fuente. En esta sección solo definimos los parámetros de ejecución.

| Parámetro | Valor | Significado |
|---|---|---|
| `SEMILLA` | 42 | Semilla de todos los procesos aleatorios |
| `N_BOOT` | 1000 | Remuestreos *bootstrap* para los intervalos de confianza del RMSE |
| `UMBRAL_ASIMETRIA` | 1.0 | Una variable se transforma si su asimetría en entrenamiento supera este valor en valor absoluto (sección 8) |
| `USAR_ZOOBOT` | `"auto"` | La extracción con Zoobot (12.4) solo corre si hay GPU |

In [ ]:
import os
env = os.environ.get          # las variables AV2_* solo se usan en la prueba local reducida

SEMILLA = 42
N_BOOT = int(env("AV2_N_BOOT", "1000"))
UMBRAL_ASIMETRIA = 1.0
USAR_ZOOBOT = env("AV2_ZOOBOT", "auto")      # "auto", "si" o "no"
OBJETIVOS = ["log_mstar", "log_sfr", "d4000"]
NOMBRES = {"log_mstar": "log M* [dex]", "log_sfr": "log SFR [dex]", "d4000": "D4000"}
# Referencias de la red convolucional (validación) y de Katachi publicado (prueba), RMSE
CNN = {"log_mstar": 0.224, "log_sfr": 0.631, "d4000": 0.075}
KATACHI = {"log_mstar": 0.222, "log_sfr": 0.668, "d4000": 0.100}

## 3. Entorno

Descargamos el código del repositorio del equipo, de modo que cada resultado quede ligado a una versión concreta, e instalamos las dependencias que Colab no incluye: `photutils`, `dustmaps` y `factor_analyzer`.

In [ ]:
import sys, json, time, subprocess, platform, warnings
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/erteck/InferenciaPropiedadesFisicas.git"
REPO_DIR = Path(env("AV2_REPO_DIR", "/content/InferenciaPropiedadesFisicas"))

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "photutils", "dustmaps", "factor_analyzer"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q"], check=True)

sys.path.insert(0, str(REPO_DIR))
COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Python", platform.python_version(), "· commit del código:", COMMIT[:12])

In [ ]:
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
from scipy import stats
from IPython.display import display

from src.caracteristicas_desi import config as FE
from src.caracteristicas_desi.diccionario import DICCIONARIO, grupo
from src.caracteristicas_desi.fisica import agregar_variables_fisicas, ebv_sfd
from src.caracteristicas_desi.fotometria import medir_fotometria
from src.caracteristicas_desi.pipeline import medir_cache
from src.caracteristicas_desi.transformaciones import TransformadorAsimetria
from src.katachi_fits import config as KC
from src.katachi_fits.io_guard import WriteGuard, sha256

pd.set_option("display.precision", 3, "display.max_columns", 40, "display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
rng = np.random.default_rng(SEMILLA)

def rmse(y, p):
    return float(np.sqrt(np.mean((np.asarray(y) - np.asarray(p)) ** 2)))

def rmse_ic(y, p, n=N_BOOT, seed=0):
    # Intervalo de confianza del 95 % del RMSE por bootstrap de galaxias.
    y, p = np.asarray(y), np.asarray(p)
    i = np.random.default_rng(seed).integers(0, len(y), (n, len(y)))
    return np.percentile(np.sqrt(((y[i] - p[i]) ** 2).mean(1)), [2.5, 97.5])

### 3.1 Dónde están los datos

| Entrada (solo lectura) | Ubicación en Drive |
|---|---|
| Catálogo maestro del equipo (etiquetas, calidad, particiones) | `descarga_22fb149046f162f3/catalogo_candidato_9661_galaxias.csv` |
| Caché de recortes de 300 × 300 px (78.6″) de los FITS | `resultados_katachi_fits/cache/desi/recorte300/` |
| Catálogo auxiliar (morfología y masas de otros catálogos) | `Datos/inferencia.fits`, en el repositorio |

La caché se generó con el cuaderno de Katachi a partir de los FITS. Si no existe, la siguiente celda la construye con el mismo código (unos 15 minutos). El código no permite escribir fuera de `resultados_fe/`.

In [ ]:
BASE = Path(env("AV2_BASE", "/content/drive/MyDrive"))     # "Mi unidad" de Google Drive
DESCARGA = BASE / KC.DOWNLOAD_DIRNAME
CACHE = BASE / KC.RESULTS_DIRNAME / "cache" / "desi" / "recorte300"
RESULTADOS = BASE / "resultados_fe"

NOMBRES_CATALOGO = ("catalogo_candidato_9661_galaxias.csv", "catalogo_candidatas_original.csv")
CATALOGO = next((p for n in NOMBRES_CATALOGO for p in DESCARGA.glob(f"**/{n}")), None)
assert CATALOGO is not None, f"No se encontró el catálogo del equipo en {DESCARGA}"
guard = WriteGuard(RESULTADOS, [DESCARGA, CACHE])
guard.mkdir(RESULTADOS)

SHA_CAT = sha256(CATALOGO)
print("Catálogo:", CATALOGO.name, "· ¿versión acordada?", SHA_CAT == KC.TEAM_CATALOG_SHA256)
assert SHA_CAT == KC.TEAM_CATALOG_SHA256 or env("AV2_BASE"), "El catálogo no es la versión acordada"

if not any(CACHE.glob("chunk_*.npz")):
    print("No hay caché de recortes: se construye a partir de los FITS.")
    from src.katachi_fits import catalog as KCat
    from src.katachi_fits.cache import build_cache
    from src.katachi_fits.config import FitsConfig
    plan = KCat.load_plan(DESCARGA / "plan.csv", DESCARGA / "fits")
    equipo = pd.read_csv(CATALOGO, dtype={"mangaid": str})
    plan = plan[plan.mangaid.isin(equipo.loc[equipo.QCFLAG_original == 0, "mangaid"])]
    guard_k = WriteGuard(BASE / KC.RESULTS_DIRNAME, [DESCARGA])
    build_cache(plan, "ruta_local", CACHE, guard_k, FitsConfig(), chunk=500, workers=24)
print("Caché:", CACHE, "·", len(list(CACHE.glob("chunk_*.npz"))), "bloques")
print("Resultados:", RESULTADOS)

## 4. Datos y selección de la muestra

**Selección.** Usamos las galaxias con `QCFLAG = 0`, es decir, aquellas cuyo ajuste espectral en Pipe3D (Sánchez et al. 2022) pasó todos los controles de calidad. Es el criterio que acordamos con los asesores. Conservamos la partición del proyecto (`split_proyecto`).

**Etiquetas.** log M\*, log SFR y D4000 de Pipe3D, **sin transformar**: log M\* y log SFR ya son logaritmos, y D4000 tiene una distribución bimodal (galaxias jóvenes y viejas) que ninguna transformación monótona vuelve normal. Además, de esta forma los resultados quedan en las mismas unidades que los de la red y Katachi.

**Variables que no entran al modelo.** Las excluimos como parte de la selección de datos, porque usarlas implicaría **fuga de información** (*data leakage*): el modelo recibiría como entrada la respuesta o una versión de ella.

| Variable | Fuente | Motivo |
|---|---|---|
| `LogMass`, `log_SFR_Ha`, `log_SFR_ssp`, `log_age_mean_LW`, `log_ZH_mean_LW`, `vel_sigma_Re` | Pipe3D | Salen del mismo ajuste espectral que las etiquetas |
| `LogMassNSA`, `LogMassTaylor`, `nsa_sersic_mass` | NSA y Taylor | Son otras estimaciones de la masa. Se usan **solo para validar** nuestras mediciones |
| `error_log_mstar`, `error_log_sfr`, `error_d4000` | Pipe3D | Describen la etiqueta, no la galaxia |
| `QCFLAG` | Pipe3D | Es constante (0) en la muestra |
| `mangaid`, RA, Dec | Catálogo | Son identificadores. Con las coordenadas en bruto un modelo podría memorizar zonas del cielo; la información útil de la posición (telescopio y extinción) la construimos aparte |

In [ ]:
cat = pd.read_csv(CATALOGO, dtype={"mangaid": str})
print("Catálogo completo:", len(cat), "galaxias")
cat = cat[cat.QCFLAG_original == 0]
datos = cat.rename(columns={"RA_Pipe3D": "ra", "DEC_Pipe3D": "dec", "split_proyecto": "particion", "redshift": "z"})[
    ["mangaid", "ra", "dec", "z", "particion", *OBJETIVOS]].reset_index(drop=True)

# Catálogo auxiliar del repositorio: -999 marca valores ausentes.
aux = Table.read(REPO_DIR / "Datos" / "inferencia.fits").to_pandas()
aux["mangaid"] = aux.MANGAID.map(lambda x: x.decode() if isinstance(x, bytes) else x).str.strip()
aux = aux[["mangaid", "T_11", "PETRO_TH90", "C", "A", "LogMassNSA", "nsa_sersic_ba"]].replace(-999.0, np.nan)
aux = aux.rename(columns={"C": "C_catalogo", "A": "A_catalogo"})
datos = datos.merge(aux, on="mangaid", how="left", validate="one_to_one")

print("Con QCFLAG = 0:", len(datos), "·", datos.particion.value_counts().to_dict())
print("Sin redshift:", datos.z.isna().sum(), "· sin T_11:", datos.T_11.isna().sum())

## 5. Construcción de características

Sobre cada recorte (3 bandas, 300 × 300 píxeles, 0.262″/px, en nanomaggies) medimos las cantidades que se usan habitualmente en astronomía extragaláctica para describir galaxias. En lugar de proponer variables nuevas, partimos de definiciones publicadas, con interpretación física y comparables con otros catálogos. Todas se miden en el **mismo recorte que recibe la red**, para que la comparación de la sección 13 sea equivalente.

### 5.1 Cómo se mide cada galaxia

| Paso | Qué se hace | Por qué |
|---|---|---|
| Cielo residual | Mediana y desviación con recorte σ en las esquinas del recorte | DESI ya resta el cielo, pero queda un residuo que afecta las magnitudes de las galaxias débiles |
| Máscara de la galaxia | Umbral de 2σ sobre la banda r suavizada; componente que contiene el centro, sin huecos | Es la máscara que elegimos en la evaluación de máscaras del equipo (recupera 0.999 de la luz con 2 % de contaminación) |
| Vecinos | Fuentes detectadas a 3σ fuera de la galaxia; sus píxeles se reemplazan por el píxel simétrico respecto al centro | Una estrella o galaxia vecina aumentaría el flujo y alteraría el color. El relleno aprovecha que las galaxias son aproximadamente simétricas |
| Radio de Petrosian R_P | Radio donde el brillo del anillo [0.8R, 1.25R] es 0.2 veces el brillo medio interior | Es la definición de SDSS (Blanton et al. 2001). No depende de la profundidad de la imagen ni de la distancia, y la apertura de 2 R_P contiene casi todo el flujo |
| Flujo y magnitud | Flujo dentro de 2 R_P **medido en r** y aplicado igual en g, r y z | Con la misma apertura en las tres bandas, el color compara la misma región de la galaxia |
| Radios R20, R50, R80, R90 | Radios que contienen esa fracción del flujo de Petrosian | Tamaño aparente y concentración |
| Morfología | C, A, S (Conselice 2003), Gini y M20 (Lotz et al. 2004), b/a | Distribución de la luz: bulbo frente a disco, perturbaciones, formación estelar grumosa |

Para la morfología seguimos las definiciones y parámetros de `statmorph` (Rodriguez-Gomez et al. 2019), con dos diferencias: el término de ruido de A y S se calcula a partir de σ del cielo en lugar de medirse en una caja de fondo, y todas las aperturas son circulares, igual que el `PETRO_TH90` de NSA con el que comparamos.

El código de medición está en `src/caracteristicas_desi/` y se verificó con 18 pruebas automáticas sobre galaxias sintéticas de Sérsic, cuyos radios, flujos y concentraciones se conocen analíticamente (`tests/caracteristicas_desi/`). La siguiente celda mide las 9,303 galaxias en paralelo y guarda cada bloque al terminar.

In [ ]:
t0 = time.time()
medidas, ESTAMPAS, ids_estampas = medir_cache(CACHE, RESULTADOS / "mediciones", guard,
                                              ids_validos=set(datos.mangaid), workers=os.cpu_count())
print(f"Medidas: {len(medidas)} galaxias en {time.time() - t0:.0f} s")
print("Estado:", medidas.estado.value_counts().to_dict())
fallos = medidas[medidas.estado != "ok"]
if len(fallos):
    display(fallos[["mangaid", "motivo"]].head(10))

### 5.2 Ejemplo de medición en una galaxia

Para revisar el procedimiento tomamos una galaxia de entrenamiento de masa intermedia. A la izquierda se muestra la imagen en color (z, r, g → rojo, verde, azul; Lupton et al. 2004) con la máscara de la galaxia (blanco), los vecinos enmascarados (rojo) y la apertura de Petrosian (amarillo, 2 R_P). Al centro aparece la imagen limpia sobre la que se mide, y a la derecha la curva de crecimiento, es decir, la fracción del flujo contenida dentro de cada radio, con R50, R90 y R_P marcados.

In [ ]:
from astropy.visualization import make_lupton_rgb

def leer_recorte(mangaid):
    # Busca el recorte de una galaxia en los bloques de la caché (solo lectura).
    for f in sorted(CACHE.glob("chunk_*.npz")):
        with np.load(f) as z:
            k = np.where(z["mangaid"].astype(str) == mangaid)[0]
            if len(k):
                return z["x"][k[0]].astype(np.float32)
    raise KeyError(mangaid)

ok_tr = datos[(datos.particion == "Train") & datos.mangaid.isin(medidas.loc[medidas.estado == "ok", "mangaid"])]
ejemplo = ok_tr.iloc[(ok_tr.log_mstar - ok_tr.log_mstar.median()).abs().argsort().iloc[0]]
m = medir_fotometria(leer_recorte(ejemplo.mangaid))
n = m.limpio.shape[-1]; c = (n - 1) / 2

fig, ax = plt.subplots(1, 3, figsize=(13, 4.2))
rgb = lambda x: make_lupton_rgb(x[2], x[1], x[0], Q=8, stretch=0.05, minimum=0)
cubo_ej = leer_recorte(ejemplo.mangaid)
ax[0].imshow(rgb(cubo_ej), origin="lower")
ax[0].contour(m.mascara_gal, [0.5], colors="w", linewidths=0.8)
ax[0].contour(m.mascara_vec, [0.5], colors="r", linewidths=0.6)
ax[0].add_patch(plt.Circle((c, c), m.r_ap_px, fill=False, color="y", lw=1))
ax[0].set_title(f"{ejemplo.mangaid} · log M* = {ejemplo.log_mstar:.2f}")
ax[1].imshow(rgb(m.limpio), origin="lower"); ax[1].set_title("Cielo restado y vecinos rellenados")
p = m.perfiles[1]; R = p.radios * FE.PIXSCALE
ax[2].plot(R, p.flujo / p.F(m.r_ap_px), "k-")
for nombre, col in (("R50", "C0"), ("R90", "C1"), ("R_P", "C2")):
    ax[2].axvline(m.valores[nombre], color=col, ls="--", label=f"{nombre} = {m.valores[nombre]:.1f}″")
ax[2].set(xlim=(0, 2.5 * m.valores["R_P"]), ylim=(0, 1.1), xlabel="radio [″]", ylabel="F(<R) / F_Petrosian",
          title="Curva de crecimiento en r"); ax[2].legend()
for a in ax[:2]: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

### 5.3 Extinción galáctica y posición en el cielo

La luz de cada galaxia atraviesa el polvo de la Vía Láctea, que la atenúa y la enrojece en distinta medida según la dirección. Como este efecto no es propio de la galaxia, lo corregimos: E(B−V) se lee del mapa de Schlegel, Finkbeiner & Davis (1998) y se multiplica por los coeficientes A/E(B−V) de Legacy Surveys DR10, que son 3.214, 2.165 y 1.211 para g, r y z (Schlafly & Finkbeiner 2011).

A partir de la posición también construimos `region_norte`. Las imágenes al norte de Dec = 32.375° provienen de BASS y MzLS, y las del sur de DECam (Dey et al. 2019). Como estos telescopios tienen filtros, profundidad y resolución distintos, la variable nos permite detectar diferencias sistemáticas entre regiones (sección 10).

In [ ]:
SFD_DIR = Path(env("AV2_SFD_DIR", str(RESULTADOS / "externos" / "sfd")))
if not env("AV2_SFD_DIR"):
    guard.mkdir(SFD_DIR)
    for nombre, url in FE.SFD_URLS.items():
        guard.download(url, SFD_DIR / nombre)
datos["ebv"] = ebv_sfd(datos.ra.to_numpy(), datos.dec.to_numpy(), SFD_DIR)
datos["region_norte"] = (datos.dec >= FE.DEC_NORTE).astype(int)
print("E(B−V): mediana", round(datos.ebv.median(), 3), "· máximo", round(datos.ebv.max(), 3))
print("Galaxias en el norte (BASS/MzLS):", int(datos.region_norte.sum()), "de", len(datos))

### 5.4 Variables físicas

Con el redshift pasamos de lo **aparente** (cómo se ve la galaxia desde aquí) a lo **intrínseco** (cómo es). Esta conversión es especialmente relevante en nuestro caso: la red convolucional mostró un sesgo en masa que depende de la distancia, pues sobreestima las galaxias cercanas y subestima las lejanas. Con la imagen sola no es posible distinguir una galaxia pequeña y cercana de una grande y lejana, y el redshift resuelve esa ambigüedad.

| Variable | Construcción | Fuente |
|---|---|---|
| Magnitudes corregidas `m_b0`, colores `g_r`, `r_z`, `g_z` | m − A_b · E(B−V) | Schlafly & Finkbeiner 2011 |
| Gradientes de color `delta_g_r`, `delta_r_z` | Color dentro de R50 menos color entre R50 y R90 | — |
| Brillo superficial `mu50_b` | m + 2.5 log10(2π R50²) | Blanton et al. 2001 |
| Módulo de distancia `DM` y magnitudes absolutas `M_b` | ΛCDM plana, H0 = 73 km/s/Mpc, Ωm = 0.3 | La cosmología de Pipe3D (Sánchez et al. 2022), para que todo sea coherente con las etiquetas |
| Luminosidad `log_L_r` | 0.4 (4.65 − M_r) | M☉,r = 4.65 (Willmer 2018) |
| Tamaño físico `log_R50_kpc` y densidad `log_SigmaL` | Escala angular a la distancia de la galaxia | — |
| Masa fotométrica `logM_ebrova` | 0.673 M_g − 1.108 M_r + 0.996 | Ebrová et al. 2025, calibrada para Legacy Surveys |
| Masa por color `logM_bell` | log L_r − 0.306 + 1.097 (g − r) | Bell et al. 2003 |
| Densidad de masa `log_SigmaM` | logM − log(2π R50²) | Relacionada con el cese de la formación estelar |

**Corrección K.** No la aplicamos. Esta corrección compensa que, a mayor redshift, cada filtro observa una región más azul del espectro. A la mediana de MaNGA (z ≈ 0.04) equivale a unas centésimas de magnitud en g y r, por lo que conservamos el redshift como variable para que el modelo absorba el efecto restante.

In [ ]:
df = datos.merge(medidas, on="mangaid", how="inner", validate="one_to_one")
df = agregar_variables_fisicas(df)
df["apertura_truncada"] = df.apertura_truncada.astype(float)
df = df.replace([np.inf, -np.inf], np.nan)
CONSTRUIDAS = [v for v in DICCIONARIO.variable if v in df.columns]
print(f"{len(df)} galaxias × {len(CONSTRUIDAS)} características construidas")

### 5.5 Validación de las mediciones

Antes de usar las características verificamos que midan lo que deben. Para ello comparamos tres mediciones con catálogos independientes, solo en entrenamiento:

- **Tamaño:** nuestro R90 frente al `PETRO_TH90` de NSA, medido con imágenes SDSS.
- **Concentración:** nuestra C frente a la del catálogo auxiliar.
- **Masa:** nuestras dos masas fotométricas frente a la de Pipe3D (la etiqueta) y a la de NSA.

Resumimos la dispersión con σ_NMAD = 1.4826 × mediana(|Δ − mediana(Δ)|), que es poco sensible a valores extremos.

In [ ]:
tr = df.particion == "Train"
def nmad(x):
    x = x[np.isfinite(x)]; return 1.4826 * np.median(np.abs(x - np.median(x)))

pares = [("R90", "PETRO_TH90", "R90 [″]"), ("C", "C_catalogo", "C"),
         ("logM_ebrova", "log_mstar", "log M* [dex]"), ("logM_bell", "log_mstar", "log M* [dex]")]
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
filas = []
for a, (x, y, et) in zip(ax, pares):
    d = df.loc[tr, [x, y]].dropna()
    dif = d[x] - d[y] if "logM" in x or x == "C" else np.log10(d[x] / d[y])
    filas.append({"medición": x, "referencia": y, "N": len(d), "Spearman ρ": stats.spearmanr(d[x], d[y])[0],
                  "desplazamiento mediano": np.median(dif), "σ_NMAD": nmad(dif.to_numpy())})
    a.hexbin(d[y], d[x], gridsize=50, mincnt=1, bins="log", cmap="viridis",
             xscale="log" if x == "R90" else "linear", yscale="log" if x == "R90" else "linear")
    lim = [np.nanpercentile(d.values, 0.5), np.nanpercentile(d.values, 99.5)]
    a.plot(lim, lim, "r--", lw=1); a.set(xlabel=f"{y}", ylabel=f"{x}", title=et)
plt.tight_layout(); plt.show()
validacion = pd.DataFrame(filas)
nsa = df.loc[tr, ["logM_ebrova", "LogMassNSA"]].dropna()
validacion.loc[len(validacion)] = ["logM_ebrova", "LogMassNSA", len(nsa), stats.spearmanr(nsa.iloc[:, 0], nsa.iloc[:, 1])[0],
                                   np.median(nsa.iloc[:, 0] - nsa.iloc[:, 1]), nmad((nsa.iloc[:, 0] - nsa.iloc[:, 1]).to_numpy())]
display(validacion)

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*
Al interpretar el desplazamiento en masa hay que considerar la función inicial de masa (IMF) de cada estimación: Ebrová et al. y Pipe3D usan la de Salpeter, mientras que NSA usa la de Chabrier, que produce masas unos 0.25 dex menores (Ebrová et al. 2025).

### 5.6 Diccionario de características

Documentamos cada característica con su definición, unidades, motivo físico y referencia, y guardamos el diccionario en `resultados_fe/diccionario_caracteristicas.csv`.

Agrupamos las características en cuatro tipos, que usamos para formar los conjuntos de la sección 13:

| Grupo | Qué contiene |
|---|---|
| **aparente** | Lo que se mide en la imagen sin conocer la distancia |
| **intrínseca** | Lo que requiere el redshift |
| **estructura** | Tamaño y distribución de la luz |
| **contexto** | Condiciones de observación, que pueden introducir sesgos |

El ángulo de posición se mide, pero **no** lo usamos como característica, porque la orientación de una galaxia en el cielo no tiene relación con sus propiedades físicas.

In [ ]:
display(DICCIONARIO.groupby("grupo").variable.count().rename("características"))
display(DICCIONARIO)
guard.write(RESULTADOS / "diccionario_caracteristicas.csv", lambda t: DICCIONARIO.to_csv(t, index=False), allow_replace=True)

## 6. Calidad de los datos: valores ausentes

Algunas mediciones no se pueden calcular en todas las galaxias:

- **R_P no converge** en las galaxias más grandes que el recorte.
- Una galaxia **no tiene redshift**, por lo que tampoco tiene variables intrínsecas.
- En galaxias muy débiles, el **flujo de alguna banda puede ser nulo o negativo**, y entonces la magnitud queda indefinida.

**Decisión.** No eliminamos galaxias, para trabajar con la misma muestra que la red. Imputamos cada valor ausente con la **mediana de entrenamiento** de la variable, que es poco sensible a las colas, y conservamos `apertura_truncada` como indicador de que la medición es menos confiable. El imputador se ajusta solo con entrenamiento y forma parte del pipeline (sección 9.3).

In [ ]:
ausentes = df.loc[df.particion != "Test", CONSTRUIDAS].isna().mean().mul(100).sort_values(ascending=False)
display(ausentes[ausentes > 0].round(2).rename("% ausente (Train + Validation)").to_frame())
print("R_P no converge:", int((~df.petro_converge.fillna(False).astype(bool)).sum()),
      "· apertura truncada:", int(df.apertura_truncada.sum()), "· fallos de medición:", int((df.estado != "ok").sum()))

## 7. Codificación y discretización

Desde aquí trabajamos con entrenamiento, para ajustar, y con validación, para medir.

### 7.1 Codificación de variables categóricas

| Variable | Categorías | Técnica | Por qué |
|---|---|---|---|
| `region_norte` | Norte (BASS/MzLS), sur (DECam) | One-hot binaria: una columna 0/1 | Son dos categorías sin orden; una sola columna es suficiente y evita la colinealidad perfecta (la *trampa de las variables ficticias*) |
| `T_11` | Tipo morfológico en la escala T de de Vaucouleurs (−5 elíptica … 10 irregular) | **Ordinal** frente a **one-hot** de clases agrupadas | El tipo morfológico tiene un orden físico (de esferoides a discos cada vez más dominantes y con más formación estelar). Comparamos si ese orden es suficiente o si conviene tratar cada clase por separado |

`T_11` proviene del catálogo auxiliar, cuya clasificación se hizo con imágenes de SDSS y no con las nuestras. Por eso no la incluimos en el conjunto principal y la evaluamos aparte en la sección 13, para medir cuánto aporta un dato externo. A las 6 galaxias sin `T_11` les asignamos la moda de entrenamiento.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer

M = df[df.particion.isin(["Train", "Validation"])].reset_index(drop=True)
tr, va = (M.particion == "Train").to_numpy(), (M.particion == "Validation").to_numpy()
Y = {t: M[t].to_numpy() for t in OBJETIVOS}
print("Entrenamiento:", tr.sum(), "· validación:", va.sum())

# T_11: ordinal (el propio valor T) y one-hot de clases agrupadas.
moda_t = M.loc[tr, "T_11"].mode()[0]
t11 = M.T_11.fillna(moda_t)
clases_t = pd.cut(t11, [-10, -3.5, 0.5, 3.5, 7.5, 12], labels=["E", "S0", "Sa-Sb", "Sbc-Sd", "Sdm-Irr"])
T11_ORD = t11.to_frame("T_11")
clases_t = clases_t.astype(str).to_frame("T")
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore").fit(clases_t[tr])
T11_OH = pd.DataFrame(ohe.transform(clases_t), columns=ohe.get_feature_names_out())
display(pd.crosstab(clases_t["T"][tr], M.loc[tr, "region_norte"].map({0: "sur", 1: "norte"}), margins=True))

### 7.2 Discretización

Al discretizar una variable continua se pierde información, por lo que **no** discretizamos ninguna característica de entrada del modelo. La discretización solo la usamos en tres casos:

1. **Intervalos de redshift** (5 cuantiles de entrenamiento), para analizar si los errores dependen de la distancia.
2. **Clases físicas de las etiquetas**, necesarias para las pruebas de chi-cuadrado y ANOVA de clasificación de la sección 11:
   - formadora frente a apagada, con log sSFR = log SFR − log M\* < −10.8 (el corte de Katachi);
   - población joven frente a vieja, con D4000 = 1.5.
3. **Características en 10 intervalos de cuantil**, solo para chi-cuadrado, que requiere conteos por categoría.

La siguiente tabla corresponde al primer caso: el sesgo de nuestra masa fotométrica en cada intervalo de redshift.

In [ ]:
kz = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy="quantile").fit(M.loc[tr, ["z"]].fillna(M.z.median()))
M["z_bin"] = kz.transform(M[["z"]].fillna(M.z.median())).astype(int)
bordes = kz.bin_edges_[0]
tabla_z = (M[tr].assign(sesgo_ebrova=M.logM_ebrova - M.log_mstar)
           .groupby("z_bin").agg(N=("z", "size"), z_min=("z", "min"), z_max=("z", "max"),
                                 log_mstar_medio=("log_mstar", "mean"), sesgo_ebrova=("sesgo_ebrova", "median")))
display(tabla_z)

# Clases físicas (solo para pruebas de filtro; nunca son características)
M["apagada"] = ((M.log_sfr - M.log_mstar) < FE.CORTE_SSFR).astype(int)
M["vieja"] = (M.d4000 > FE.CORTE_D4000).astype(int)
print("Fracción apagadas:", round(M.loc[tr, "apagada"].mean(), 3), "· fracción viejas:", round(M.loc[tr, "vieja"].mean(), 3))

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

## 8. Transformación de variables asimétricas

Muchas cantidades astronómicas son asimétricas: los flujos y los tamaños tienen colas largas hacia valores altos, y la asimetría A se concentra cerca de 0 con unas cuantas galaxias perturbadas muy por encima. Una variable con cola larga genera tres problemas:

- domina la media y la varianza de un escalador estándar;
- la regresión lineal se ajusta a los valores extremos;
- los métodos basados en correlación lineal, como PCA y el análisis factorial, la representan mal.

Este es el **riesgo de características sesgadas** al que se refiere el objetivo 2.4.

**Regla de transformación.** La definimos antes de ver resultados: calculamos la asimetría (*skewness*) de cada característica en entrenamiento y aplicamos lo siguiente:

| Caso | Transformación |
|---|---|
| \|asimetría\| ≤ 1 | Ninguna |
| \|asimetría\| > 1 y la variable es estrictamente positiva | Se prueban **log10** y **Box-Cox** (Box & Cox 1964) y se elige la que deja la asimetría más cerca de 0. El logaritmo es la opción natural para cantidades multiplicativas, como flujos y radios |
| \|asimetría\| > 1 y la variable puede valer 0 o ser negativa | **Yeo-Johnson** (Yeo & Johnson 2000), que extiende Box-Cox a toda la recta real |
| Variables binarias | Nunca se transforman |

El umbral de 1 es el criterio habitual para considerar alta la asimetría (Bulmer 1979). El parámetro λ se estima por máxima verosimilitud en entrenamiento y se aplica sin cambios a validación (Galli 2022, cap. 3). Lo limitamos al intervalo [−3, 3] porque, en variables casi siempre nulas con unos pocos valores altos (por ejemplo, la fracción de vecinos), la máxima verosimilitud produce λ extremos que vuelven inestable la transformación fuera de entrenamiento. La columna `lambda_limitado` indica en qué variables se aplicó el límite.

In [ ]:
X_crudo = M[CONSTRUIDAS].astype(float)
asim = TransformadorAsimetria(umbral=UMBRAL_ASIMETRIA).fit(X_crudo[tr])
rep = asim.reporte_.assign(abs_antes=lambda d: d.asimetria_antes.abs()).sort_values("abs_antes", ascending=False)
print("Transformaciones elegidas:", rep.transformacion.value_counts().to_dict())
display(rep.drop(columns="abs_antes").round(3))
X_trans = asim.transform(X_crudo)

In [ ]:
peores = [v for v in rep.index if rep.loc[v, "transformacion"] != "ninguna"][:4]
fig, ax = plt.subplots(2, len(peores), figsize=(3.3 * len(peores), 6))
for j, v in enumerate(peores):
    for i, (X, et) in enumerate(((X_crudo, "antes"), (X_trans, rep.loc[v, "transformacion"]))):
        x = X.loc[tr, v].dropna().to_numpy()
        stats.probplot(x, dist="norm", plot=ax[i, j])
        ax[i, j].set_title(f"{v} · {et}\nasimetría = {stats.skew(x):.2f}", fontsize=8)
        ax[i, j].set_xlabel(""); ax[i, j].set_ylabel("")
plt.suptitle("Gráficas Q-Q contra la normal (entrenamiento): las cuatro variables más asimétricas", y=1.01)
plt.tight_layout(); plt.show()

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

## 9. Escalamiento

Las características tienen unidades y escalas muy distintas: las magnitudes rondan 15, los radios 10″, E(B−V) 0.03 y el número de fuentes vecinas 30. Los modelos basados en distancias, gradientes o penalizaciones (regresión regularizada, redes neuronales, PCA) dan más peso a las variables de mayor escala aunque no sean las más informativas. Con el escalamiento buscamos que **todas tengan un impacto comparable** en el modelo.

| Escalador | Transformación | Cuándo conviene |
|---|---|---|
| `StandardScaler` | (x − media) / desviación | Opción por defecto: centra en 0 con varianza 1, que es lo que suponen PCA, el análisis factorial y la inicialización de redes neuronales |
| `RobustScaler` | (x − mediana) / rango intercuartílico | Cuando quedan valores extremos, ya que la mediana y los cuartiles casi no cambian con ellos |
| `MinMaxScaler` | (x − mín) / (máx − mín) → [0, 1] | Cuando un método exige valores acotados o no negativos (umbral de varianza, chi-cuadrado) |

### 9.1 Elección del escalador

Para elegir, aplicamos cada escalador después de la transformación de la sección 8 y comparamos el RMSE en validación de una regresión lineal regularizada (Ridge), que es sensible a la escala de las variables.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.linear_model import Ridge

filas = []
for nombre, esc in (("StandardScaler", StandardScaler()), ("RobustScaler", RobustScaler()), ("MinMaxScaler", MinMaxScaler())):
    pre = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", esc)]).fit(X_trans[tr])
    Z = pre.transform(X_trans)
    fila = {"escalador": nombre}
    for t in OBJETIVOS:
        fila[NOMBRES[t]] = rmse(Y[t][va], Ridge(alpha=1.0).fit(Z[tr], Y[t][tr]).predict(Z[va]))
    filas.append(fila)
display(pd.DataFrame(filas).set_index("escalador").round(4))

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

### 9.2 Prueba de convergencia (objetivo 2.4)

Además de equilibrar el peso de las variables, el escalamiento **acelera la convergencia** de los algoritmos que se optimizan por gradiente. Cuando las variables tienen escalas muy distintas, la función de pérdida forma un valle alargado: el gradiente no apunta hacia el mínimo y el optimizador avanza en zigzag. Lo medimos de tres formas, con la misma muestra y en tres versiones de los datos:

1. **Número de condición** de la matriz de diseño, es decir, el cociente entre su mayor y su menor valor singular. Mientras más grande, peor condicionado está el problema.
2. **Iteraciones de `SGDRegressor`** hasta converger, y su RMSE en validación.
3. **Curva de pérdida de `MLPRegressor`**, una red neuronal pequeña.

Las tres versiones son: (a) crudas, solo imputadas; (b) imputadas y estandarizadas; (c) transformadas, imputadas y estandarizadas.

Para esta prueba usamos un subconjunto fijo de 14 características sin combinaciones lineales exactas. Con todas, la matriz sería singular, porque algunas variables son sumas exactas de otras (g − z = (g − r) + (r − z); M_b = m_b0 − DM), y el número de condición sería infinito con cualquier escalamiento. Esa redundancia la tratamos en la selección de la sección 11.

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.exceptions import ConvergenceWarning

SUB = ["m_r0", "g_r", "r_z", "R50", "C", "A", "S", "gini", "m20", "b_a", "ebv", "z", "n_fuentes", "sigma_cielo_r"]
imp = lambda: SimpleImputer(strategy="median")
versiones = {
    "(a) crudas": Pipeline([("imputar", imp())]).fit(X_crudo.loc[tr, SUB]).transform(X_crudo[SUB]),
    "(b) estandarizadas": Pipeline([("imputar", imp()), ("escalar", StandardScaler())]).fit(X_crudo.loc[tr, SUB]).transform(X_crudo[SUB]),
    "(c) transformadas + estandarizadas": Pipeline([("imputar", imp()), ("escalar", StandardScaler())]).fit(X_trans.loc[tr, SUB]).transform(X_trans[SUB]),
}
filas, curvas = [], {}
y_tr, y_va = Y["log_mstar"][tr], Y["log_mstar"][va]
for nombre, Z in versiones.items():
    cond = np.linalg.cond(np.c_[np.ones(tr.sum()), Z[tr]])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        t0 = time.time()
        sgd = SGDRegressor(max_iter=5000, tol=1e-5, random_state=SEMILLA).fit(Z[tr], y_tr)
        t_sgd = time.time() - t0
        p = sgd.predict(Z[va])
        mlp = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=SEMILLA).fit(Z[tr], y_tr)
    curvas[nombre] = mlp.loss_curve_
    filas.append({"datos": nombre, "número de condición": cond, "iteraciones SGD": sgd.n_iter_, "tiempo SGD [s]": t_sgd,
                  "RMSE SGD (validación)": rmse(y_va, p) if np.isfinite(p).all() else np.inf,
                  "RMSE MLP (validación)": rmse(y_va, mlp.predict(Z[va])), "iteraciones MLP": mlp.n_iter_})
display(pd.DataFrame(filas).set_index("datos"))

fig, ax = plt.subplots(figsize=(6, 3.5))
for nombre, cur in curvas.items():
    ax.plot(np.arange(1, len(cur) + 1), cur, label=nombre)
ax.set(yscale="log", xlabel="época", ylabel="pérdida de entrenamiento (MSE/2)", title="MLPRegressor · log M*")
ax.legend(); plt.tight_layout(); plt.show()

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

### 9.3 Pipeline de preparación

Las decisiones de las secciones 6, 8 y 9 quedan encadenadas en un solo `Pipeline` de scikit-learn:

**transformación por asimetría → imputación con la mediana → `StandardScaler`**

El pipeline se ajusta una sola vez con entrenamiento y se guarda en `resultados_fe/pipeline_preparacion.joblib`. De esta forma, en la siguiente fase se pueden aplicar exactamente las mismas operaciones a cualquier galaxia sin volver a ajustar nada, lo que corresponde a la tarea de CRISP-ML(Q) de construir pipelines de transformación reproducibles.

In [ ]:
import joblib
PREP = Pipeline([("asimetria", TransformadorAsimetria(umbral=UMBRAL_ASIMETRIA)),
                 ("imputar", SimpleImputer(strategy="median")),
                 ("escalar", StandardScaler())]).fit(X_crudo[tr])
Z = pd.DataFrame(PREP.transform(X_crudo), columns=CONSTRUIDAS)
guard.write(RESULTADOS / "pipeline_preparacion.joblib", lambda t: joblib.dump(PREP, t), allow_replace=True)
print("Media en entrenamiento (debe ser ≈ 0):", float(np.abs(Z[tr].mean()).max().round(6)),
      "· desviación (≈ 1):", float(Z[tr].std(ddof=0).mean().round(4)))

## 10. Sesgos por condiciones de observación

Consideramos que una característica está **sesgada** cuando su valor depende de cómo se observó la galaxia y no solo de cómo es; un modelo que aprende de ella traslada ese sesgo a sus predicciones. Revisamos dos fuentes posibles:

1. **El telescopio.** Las galaxias del norte se observaron con BASS y MzLS, y las del sur con DECam. Sus imágenes tienen filtros, profundidad y resolución distintos, y una red entrenada con imágenes de una región no puede aplicarse directamente a la otra (Ye et al. 2025). Comparamos cada característica entre regiones con la **diferencia estandarizada de medias** (SMD: diferencia de medias dividida entre la desviación conjunta; un valor > 0.1 indica desequilibrio, según Austin 2009) y la prueba de **Kolmogorov-Smirnov**. Para no confundir el efecto del telescopio con diferencias en la población, hacemos la comparación dentro de intervalos de masa estelar.
2. **La distancia.** Una galaxia más lejana ocupa menos píxeles y la PSF suaviza más su perfil, por lo que su concentración y su Gini aparentes disminuyen aunque la galaxia sea la misma. Medimos la correlación de cada característica estructural con el redshift **a masa fija**.

In [ ]:
D = M[tr].copy()
D["bin_m"] = pd.cut(D.log_mstar, [8, 9.5, 10, 10.5, 11, 12.5])
filas = []
for v in CONSTRUIDAS:
    if v in ("region_norte", "ebv"):      # describen la posición en el cielo por definición
        continue
    smd, ks = [], []
    for _, g in D.groupby("bin_m", observed=True):
        a, b = g.loc[g.region_norte == 1, v].dropna(), g.loc[g.region_norte == 0, v].dropna()
        if len(a) > 20 and len(b) > 20:
            s = np.sqrt((a.var() + b.var()) / 2)
            smd.append((a.mean() - b.mean()) / s if s > 0 else 0)
            ks.append(stats.ks_2samp(a, b).statistic)
    filas.append({"variable": v, "SMD norte−sur (a masa fija, mediana)": np.median(smd) if smd else np.nan,
                  "KS (mediana)": np.median(ks) if ks else np.nan})
sesgo_region = pd.DataFrame(filas).set_index("variable")
sesgo_region = sesgo_region.reindex(sesgo_region.iloc[:, 0].abs().sort_values(ascending=False).index)
display(sesgo_region.head(12).round(3))
print("Variables con |SMD| > 0.1:", int((sesgo_region.iloc[:, 0].abs() > 0.1).sum()), "de", len(sesgo_region))

In [ ]:
filas = []
for v in ["C", "C_sdss", "gini", "m20", "A", "S", "R50", "mu50_r", "b_a"]:
    rs = [stats.spearmanr(g.z, g[v], nan_policy="omit")[0] for _, g in D.groupby("bin_m", observed=True) if len(g) > 50]
    filas.append({"variable": v, "ρ(z) a masa fija, mediana": np.median(rs)})
display(pd.DataFrame(filas).set_index("variable").round(3))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
g = D[(D.log_mstar > 10) & (D.log_mstar < 10.5)]
ax[0].scatter(g.z, g.C, s=3, alpha=0.4); ax[0].set(xlabel="z", ylabel="C", title="Concentración con 10 < log M* < 10.5")
for reg, et in ((0, "sur (DECam)"), (1, "norte (BASS/MzLS)")):
    ax[1].hist(D.loc[D.region_norte == reg, "sigma_cielo_g"], bins=60, histtype="step", density=True, label=et)
ax[1].set(xlabel="σ del cielo en g [nmgy/px]", title="Profundidad de la imagen por región"); ax[1].legend()
plt.tight_layout(); plt.show()

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*
**Mitigación.** En lugar de eliminar características, le damos al modelo variables que explican el sesgo: `region_norte` y `sigma_cielo_*` describen el telescopio y la profundidad, y `z` la distancia. En la sección 13 medimos si con ellas disminuye el error.

## 11. Selección de características con métodos de filtro

Los métodos de filtro evalúan cada característica por sus propiedades estadísticas, sin entrenar el modelo final, por lo que son rápidos y no dependen de un algoritmo en particular. Los aplicamos en cascada, siempre sobre entrenamiento:

| Paso | Método | Qué elimina o mide |
|---|---|---|
| 1 | **Umbral de varianza** sobre los datos en escala [0, 1] (`MinMaxScaler`) | Variables casi constantes, que no pueden explicar diferencias entre galaxias |
| 2 | **Correlación de Spearman** con agrupamiento jerárquico, \|ρ\| > 0.95 | Redundancia: de cada grupo de variables casi idénticas (p. ej. R80 y R90, o dos magnitudes de la misma galaxia) conservamos la de mayor información mutua con las etiquetas. Spearman también detecta relaciones monótonas no lineales |
| 3 | **ANOVA F** (`f_regression`) e **información mutua** (`mutual_info_regression`) con las tres etiquetas | Relevancia lineal y no lineal para la regresión |
| 4 | **Chi-cuadrado** y **ANOVA de clasificación** (`f_classif`) contra las clases formadora/apagada y joven/vieja | Relevancia para separar poblaciones físicas. Chi-cuadrado usa las características en 10 intervalos de cuantil |
| 5 | **Curva de validación cruzada**: error frente a las k mejores del ranking combinado | Cuántas características se necesitan. Elegimos la k más pequeña cuyo error queda a menos de un error estándar del mínimo (*regla de un error estándar*; Hastie et al. 2009, §7.10) |

In [ ]:
from sklearn.feature_selection import VarianceThreshold, f_regression, mutual_info_regression, chi2, f_classif
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

Xtr_t = pd.DataFrame(SimpleImputer(strategy="median").fit(X_trans[tr]).transform(X_trans[tr]), columns=CONSTRUIDAS)

# Paso 1: umbral de varianza en escala [0, 1]
vt = VarianceThreshold(threshold=1e-3).fit(MinMaxScaler().fit_transform(Xtr_t))
var01 = pd.Series(vt.variances_, index=CONSTRUIDAS)
quedan = [v for v, k in zip(CONSTRUIDAS, vt.get_support()) if k]
print("Paso 1 · eliminadas por varianza < 0.001:", sorted(set(CONSTRUIDAS) - set(quedan)) or "ninguna")

# Información mutua media con las tres etiquetas (se usa para desempatar en el paso 2)
mi = pd.DataFrame({t: mutual_info_regression(Xtr_t[quedan], Y[t][tr], random_state=SEMILLA) for t in OBJETIVOS}, index=quedan)

# Paso 2: grupos con |ρ| > 0.95
rho = Xtr_t[quedan].corr(method="spearman").abs().fillna(0)
dist = squareform(1 - rho.to_numpy(), checks=False).clip(0)
grupos = fcluster(linkage(dist, "average"), t=0.05, criterion="distance")
redundantes, representantes = {}, []
for gid in np.unique(grupos):
    miembros = [v for v, g in zip(quedan, grupos) if g == gid]
    mejor = mi.loc[miembros].mean(1).idxmax()
    representantes.append(mejor)
    if len(miembros) > 1:
        redundantes[mejor] = [v for v in miembros if v != mejor]
print(f"Paso 2 · {len(quedan)} → {len(representantes)} variables. Grupos redundantes (se conserva la clave):")
for k, v in redundantes.items():
    print(f"   {k:16s} ⟵ {', '.join(v)}")

In [ ]:
import matplotlib.colors as mcolors
orden = [quedan[i] for i in np.argsort(grupos)]
fig, ax = plt.subplots(figsize=(11, 9.5))
im = ax.imshow(Xtr_t[orden].corr(method="spearman"), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(orden))); ax.set_xticklabels(orden, rotation=90, fontsize=6)
ax.set_yticks(range(len(orden))); ax.set_yticklabels(orden, fontsize=6)
plt.colorbar(im, ax=ax, shrink=0.7, label="ρ de Spearman"); ax.set_title("Correlación entre características (entrenamiento), ordenadas por grupo")
plt.tight_layout(); plt.show()

In [ ]:
# Pasos 3 y 4: relevancia
R_ = representantes
puntajes = pd.DataFrame(index=R_)
for t in OBJETIVOS:
    puntajes[f"F {t}"] = f_regression(Xtr_t[R_], Y[t][tr])[0]
    puntajes[f"IM {t}"] = mi.loc[R_, t]
Xbin = KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="quantile").fit_transform(Xtr_t[R_])
for clase in ("apagada", "vieja"):
    yc = M.loc[tr, clase].to_numpy()
    puntajes[f"χ² {clase}"] = chi2(Xbin, yc)[0]
    puntajes[f"F clas. {clase}"] = f_classif(Xtr_t[R_], yc)[0]
puntajes["rango medio"] = puntajes.rank(ascending=False).mean(1)
puntajes = puntajes.sort_values("rango medio")
RANKING = puntajes.index.tolist()
display(puntajes.round(3))

### 11.1 Número de características

Tomamos las k primeras características del ranking combinado y medimos el error con validación cruzada de 5 particiones **dentro de entrenamiento**, para no usar el conjunto de validación en esta decisión. Usamos `HistGradientBoostingRegressor`, que capta relaciones no lineales y no depende de la escala. Para promediar las tres etiquetas, dividimos cada RMSE entre la desviación estándar de su etiqueta.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold

def hgb():
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.08, random_state=SEMILLA)

ks = sorted({k for k in (3, 5, 8, 12, 16, 24, len(RANKING)) if k <= len(RANKING)})
kf = KFold(5, shuffle=True, random_state=SEMILLA)
curva = []
for k in ks:
    cols = RANKING[:k]
    por_fold = np.zeros(5)
    for t in OBJETIVOS:
        y = Y[t][tr]; s = y.std()
        for f, (a, b) in enumerate(kf.split(Xtr_t)):
            p = hgb().fit(Xtr_t.iloc[a][cols], y[a]).predict(Xtr_t.iloc[b][cols])
            por_fold[f] += rmse(y[b], p) / s / len(OBJETIVOS)
    curva.append({"k": k, "error normalizado": por_fold.mean(), "error estándar": por_fold.std(ddof=1) / np.sqrt(5)})
curva = pd.DataFrame(curva).set_index("k")
mejor = curva["error normalizado"].idxmin()
limite = curva.loc[mejor, "error normalizado"] + curva.loc[mejor, "error estándar"]
K_SEL = int(curva.index[curva["error normalizado"] <= limite].min())
SELECCION = RANKING[:K_SEL]
display(curva.round(4))
print(f"Mínimo en k = {mejor}; regla de un error estándar → k = {K_SEL}")
print("Características seleccionadas:", SELECCION)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(curva.index, curva["error normalizado"], curva["error estándar"], marker="o", capsize=3)
ax.axhline(limite, color="gray", ls=":", label="mínimo + 1 error estándar"); ax.axvline(K_SEL, color="C3", ls="--", label=f"k elegido = {K_SEL}")
ax.set(xscale="log", xlabel="número de características (k)", ylabel="RMSE / σ(y), media de 3 etiquetas",
       title="Validación cruzada en entrenamiento"); ax.legend(); plt.tight_layout(); plt.show()
guard.write_json(RESULTADOS / "seleccion.json", {"ranking": RANKING, "k": K_SEL, "seleccion": SELECCION,
                                                  "redundantes": redundantes, "commit": COMMIT}, allow_replace=True)

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

## 12. Extracción de características

La selección conserva variables originales; la extracción, en cambio, las **combina** en unas pocas variables nuevas que concentran la información, a costa de parte de la interpretabilidad. Aplicamos dos técnicas sobre las características y dos sobre los píxeles:

| Técnica | Sobre | Qué produce |
|---|---|---|
| **PCA** | Las características transformadas y estandarizadas | Combinaciones lineales ortogonales, ordenadas por la varianza que explican |
| **Análisis factorial** con rotación varimax | Las mismas | Factores latentes que explican las correlaciones; la rotación los hace interpretables |
| **Eigengalaxias** (Uzeirbegovic et al. 2020) | Estampas de 32 × 32 px de cada galaxia, alineadas por su eje mayor y escaladas a ±2 R_P | Una base de imágenes; cada galaxia se resume con sus coeficientes |
| **Zoobot + PCA** (Walmsley et al. 2023), *opcional* | La imagen en color | Representaciones de una red preentrenada con millones de clasificaciones de Galaxy Zoo |

### 12.1 Análisis de componentes principales

Aplicamos PCA a las características transformadas y estandarizadas, y conservamos los componentes necesarios para explicar el 95 % de la varianza. La gráfica de cargas muestra qué variables combina cada componente.

In [ ]:
from sklearn.decomposition import PCA, FactorAnalysis

Zc = Z[CONSTRUIDAS]
pca = PCA(random_state=SEMILLA).fit(Zc[tr])
var_acum = np.cumsum(pca.explained_variance_ratio_)
N_PCA = int(np.searchsorted(var_acum, 0.95) + 1)
print(f"Componentes para explicar el 95 % de la varianza: {N_PCA} de {len(CONSTRUIDAS)}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(1, 21), pca.explained_variance_ratio_[:20], label="por componente")
ax[0].plot(range(1, 21), var_acum[:20], "C3o-", label="acumulada")
ax[0].axhline(0.95, color="gray", ls=":"); ax[0].set(xlabel="componente", ylabel="fracción de varianza", title="Gráfica de sedimentación")
ax[0].legend()
cargas = pd.DataFrame(pca.components_[:6].T, index=CONSTRUIDAS, columns=[f"CP{i + 1}" for i in range(6)])
top = cargas.abs().max(1).sort_values(ascending=False).index[:22]
im = ax[1].imshow(cargas.loc[top], cmap="RdBu_r", vmin=-0.5, vmax=0.5, aspect="auto")
ax[1].set_yticks(range(len(top))); ax[1].set_yticklabels(top, fontsize=7)
ax[1].set_xticks(range(6)); ax[1].set_xticklabels(cargas.columns); ax[1].set_title("Cargas de los seis primeros componentes")
plt.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.show()
PCA_TR = pca.transform(Zc)[:, :N_PCA]

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

### 12.2 Análisis factorial

El análisis factorial supone que las características observadas son combinaciones de unos pocos factores latentes más un ruido propio de cada variable. Antes de aplicarlo verificamos que sea pertinente:

- **Prueba de esfericidad de Bartlett:** comprueba que la matriz de correlaciones no sea la identidad, es decir, que las variables no sean independientes.
- **Índice KMO** (Kaiser 1974): mide qué parte de la correlación es compartida; un valor mayor que 0.6 se considera adecuado.

Elegimos el número de factores con el **análisis paralelo** de Horn (1965), que conserva los factores cuyos autovalores superan a los de datos aleatorios del mismo tamaño. Usamos el subconjunto sin redundancias del paso 2 de la selección, porque las variables casi idénticas vuelven singular la matriz de correlaciones.

In [ ]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

Zf = Z[representantes]
chi2_b, p_b = calculate_bartlett_sphericity(Zf[tr])
_, kmo = calculate_kmo(Zf[tr])
print(f"Bartlett: χ² = {chi2_b:.0f}, p = {p_b:.2g} · KMO global = {kmo:.3f}")

auto = np.sort(np.linalg.eigvalsh(np.corrcoef(Zf[tr].T)))[::-1]
azar = np.mean([np.sort(np.linalg.eigvalsh(np.corrcoef(rng.standard_normal(Zf[tr].shape).T)))[::-1] for _ in range(20)], 0)
N_FA = int(np.argmax(auto < azar)) or 1
print("Factores retenidos por análisis paralelo:", N_FA)

fa = FactorAnalyzer(n_factors=N_FA, rotation="varimax" if N_FA > 1 else None, method="minres").fit(Zf[tr])
cargas_fa = pd.DataFrame(fa.loadings_, index=representantes, columns=[f"F{i + 1}" for i in range(N_FA)])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1, 1.3]})
ax[0].plot(range(1, len(auto) + 1), auto, "o-", label="datos"); ax[0].plot(range(1, len(azar) + 1), azar, "s--", label="aleatorio")
ax[0].axvline(N_FA + 0.5, color="C3", ls=":"); ax[0].set(xlabel="factor", ylabel="autovalor", title="Análisis paralelo"); ax[0].legend()
im = ax[1].imshow(cargas_fa, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax[1].set_yticks(range(len(representantes))); ax[1].set_yticklabels(representantes, fontsize=6)
ax[1].set_xticks(range(N_FA)); ax[1].set_xticklabels(cargas_fa.columns); ax[1].set_title("Cargas tras rotación varimax")
plt.colorbar(im, ax=ax[1]); plt.tight_layout(); plt.show()
print("Varianza explicada por factor:", np.round(fa.get_factor_variance()[1], 3))
FA_TR = fa.transform(Zf)

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

### 12.3 Eigengalaxias

Las características anteriores resumen la imagen con fórmulas que elegimos nosotros. Con las eigengalaxias son los datos los que definen la representación: un PCA sobre los píxeles de miles de galaxias encuentra las *imágenes base* que, combinadas, mejor reconstruyen cada galaxia (Uzeirbegovic, Geach & Kaviraj 2020).

Para que describan **forma y color**, y no tamaño ni orientación (que ya tenemos como características), preparamos cada galaxia así:

1. se rota para que su eje mayor quede horizontal;
2. se reescala para que la estampa cubra ±2 R_P y se reduce a 32 × 32 píxeles por banda;
3. se divide entre su flujo medio en r, lo que conserva el color, y se comprime con asinh.

Cada galaxia queda descrita por 3,072 valores (3 × 32 × 32), que el PCA reduce a unas decenas de coeficientes.

In [ ]:
pos = {m: i for i, m in enumerate(ids_estampas)}
E = ESTAMPAS[[pos[m] for m in M.mangaid]].reshape(len(M), -1)
validas = np.isfinite(E).all(1)
E = np.where(validas[:, None], E, 0.0)
print("Estampas válidas:", int(validas.sum()), "de", len(M))
peig = PCA(n_components=min(60, int(tr.sum()) - 1), random_state=SEMILLA).fit(E[tr & validas])
N_EIG = int(min(np.searchsorted(np.cumsum(peig.explained_variance_ratio_), 0.90) + 1, peig.n_components_))
EIG = peig.transform(E)[:, :N_EIG]
print(f"Eigengalaxias para el 90 % de la varianza: {N_EIG}")

fig, ax = plt.subplots(2, 8, figsize=(14, 3.8))
media = peig.mean_.reshape(3, 32, 32)
for j in range(8):
    if j == 0:
        img, tit = media[1], "media (r)"
    else:
        img, tit = peig.components_[j - 1].reshape(3, 32, 32)[1], f"EG{j} ({peig.explained_variance_ratio_[j - 1]:.0%})"
    ax[0, j].imshow(img, cmap="RdBu_r" if j else "gray", origin="lower"); ax[0, j].set_title(tit, fontsize=8)
    if j:
        col = peig.components_[j - 1].reshape(3, 32, 32)
        ax[1, j].imshow(col[0] - col[2], cmap="PuOr", origin="lower"); ax[1, j].set_title("g − z", fontsize=7)
for a in ax.ravel(): a.set_xticks([]); a.set_yticks([])
ax[1, 0].axis("off")
plt.suptitle("Galaxia media y primeras eigengalaxias. Arriba: banda r. Abajo: diferencia g − z (color)", y=1.02)
plt.tight_layout(); plt.show()

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

### 12.4 Zoobot (opcional, requiere GPU)

Zoobot (Walmsley et al. 2023) es una familia de redes entrenadas con las clasificaciones morfológicas de los voluntarios de Galaxy Zoo, que incluyen imágenes de DESI. Su penúltima capa produce una **representación** de la morfología (640 valores en ConvNeXt-Nano), que usamos como características extraídas y reducimos con PCA.

Como Zoobot se entrenó con imágenes en color de 8 bits y no con FITS, le damos la versión en color de Lupton et al. (2004) de cada galaxia, centrada y escalada a ±2 R_P.

In [ ]:
import torch
ZOO = None
if USAR_ZOOBOT == "si" or (USAR_ZOOBOT == "auto" and torch.cuda.is_available()):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
        import timm
        from src.caracteristicas_desi.imagenes import rgb_lupton
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        enc = timm.create_model("hf_hub:mwalmsley/zoobot-encoder-convnext_nano", pretrained=True, num_classes=0).eval().to(dev)
        rp_px = (M.set_index("mangaid").R_P / FE.PIXSCALE).to_dict()
        rep_zoo = {}
        for f in sorted(CACHE.glob("chunk_*.npz")):
            with np.load(f) as z:
                x, ids = z["x"], z["mangaid"].astype(str)
            sel = [i for i, m_ in enumerate(ids) if m_ in rp_px]
            for i0 in range(0, len(sel), 64):
                lote = sel[i0:i0 + 64]
                imgs = np.stack([rgb_lupton(x[i].astype(np.float32), rp_px[ids[i]]) for i in lote]) / 255.0
                with torch.no_grad():
                    out = enc(torch.tensor(imgs, dtype=torch.float32).permute(0, 3, 1, 2).to(dev)).cpu().numpy()
                rep_zoo.update({ids[i]: o for i, o in zip(lote, out)})
        R_zoo = np.stack([rep_zoo[m_] for m_ in M.mangaid])
        sc_zoo = StandardScaler().fit(R_zoo[tr])
        pzoo = PCA(n_components=0.90, random_state=SEMILLA).fit(sc_zoo.transform(R_zoo[tr]))
        ZOO = pzoo.transform(sc_zoo.transform(R_zoo))
        print(f"Zoobot: representación de {R_zoo.shape[1]} → {ZOO.shape[1]} componentes (90 % de la varianza)")
    except Exception as e:
        print("Zoobot no se pudo ejecutar:", type(e).__name__, str(e)[:200])
else:
    print("Sin GPU: se omite Zoobot (la comparación de la sección 13 no lo incluye).")

## 13. ¿Cuánto predicen las características?

Comparamos los conjuntos de características con dos modelos fijos, **sin ajustar hiperparámetros**, ya que lo que queremos comparar son las características y no los modelos:

- **Ridge** (regresión lineal regularizada, α = 1): es sensible a la escala y muestra cuánta información es accesible de forma lineal.
- **HistGradientBoosting**: capta relaciones no lineales e interacciones.

Ambos se ajustan con entrenamiento y se evalúan en validación. Reportamos el RMSE con su intervalo de confianza del 95 % por *bootstrap*, el número de variables, el tiempo de entrenamiento y la memoria que ocupa la matriz de entrada.

| Conjunto | Contenido |
|---|---|
| S0 | Solo el redshift (referencia mínima) |
| S1 | Fotometría aparente: magnitudes, colores, gradientes de color y brillo superficial |
| S2 | S1 + variables intrínsecas (requieren el redshift) |
| S3 | Todas las construidas (S2 + estructura + contexto) |
| S3 sin contexto | S3 sin región, profundidad ni vecinos (mide el efecto de las variables de sesgo) |
| S4 | Las seleccionadas en la sección 11 |
| S5 | Componentes principales de S3 (95 % de la varianza) |
| S6 | Factores del análisis factorial |
| S7 | Eigengalaxias |
| S8 | S4 + eigengalaxias |
| S9 | Zoobot + PCA (solo con GPU) |
| S4 + T_11 | S4 con el tipo morfológico externo, ordinal y one-hot |

In [ ]:
def estandarizar(A):
    return StandardScaler().fit(A[tr]).transform(A)

g_ap, g_in, g_es, g_cx = (grupo(n) for n in ("aparente", "intrinseca", "estructura", "contexto"))
col = lambda vs: [v for v in vs if v in CONSTRUIDAS]
CONJUNTOS = {
    "S0 · redshift": Z[["z"]].to_numpy(),
    "S1 · aparente": Z[col(g_ap)].to_numpy(),
    "S2 · + intrínseca": Z[col(g_ap + g_in)].to_numpy(),
    "S3 · todas": Z[CONSTRUIDAS].to_numpy(),
    "S3 sin contexto": Z[col(g_ap + g_in + g_es)].to_numpy(),
    f"S4 · seleccionadas ({K_SEL})": Z[SELECCION].to_numpy(),
    f"S5 · PCA ({N_PCA})": PCA_TR,
    f"S6 · factores ({N_FA})": FA_TR,
    f"S7 · eigengalaxias ({N_EIG})": estandarizar(EIG),
    "S8 · S4 + eigengalaxias": np.c_[Z[SELECCION].to_numpy(), estandarizar(EIG)],
}
if ZOO is not None:
    CONJUNTOS[f"S9 · Zoobot ({ZOO.shape[1]})"] = ZOO
CONJUNTOS["S4 + T_11 ordinal"] = np.c_[Z[SELECCION].to_numpy(), estandarizar(T11_ORD.to_numpy())]
CONJUNTOS["S4 + T_11 one-hot"] = np.c_[Z[SELECCION].to_numpy(), T11_OH.to_numpy()]

filas, PRED = [], {}
for nombre, X in CONJUNTOS.items():
    for modelo, fabrica in (("Ridge", lambda: Ridge(alpha=1.0)), ("HGB", hgb)):
        fila = {"conjunto": nombre, "modelo": modelo, "variables": X.shape[1], "memoria [MB]": X[tr].nbytes / 1e6}
        t0 = time.time()
        for t in OBJETIVOS:
            p = fabrica().fit(X[tr], Y[t][tr]).predict(X[va])
            PRED[(nombre, modelo, t)] = p
            lo, hi = rmse_ic(Y[t][va], p)
            fila[NOMBRES[t]] = rmse(Y[t][va], p)
            fila[f"IC {t}"] = f"[{lo:.3f}, {hi:.3f}]"
        fila["tiempo [s]"] = time.time() - t0
        filas.append(fila)
RES = pd.DataFrame(filas)
display(RES.set_index(["conjunto", "modelo"]).round(4))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5.2), sharey=True)
etiquetas = list(CONJUNTOS)
yy = np.arange(len(etiquetas))
for a, t in zip(ax, OBJETIVOS):
    for k, (modelo, mk) in enumerate((("Ridge", "s"), ("HGB", "o"))):
        r = RES[RES.modelo == modelo].set_index("conjunto").loc[etiquetas]
        ic = np.array([[float(s) for s in v.strip("[]").split(",")] for v in r[f"IC {t}"]])
        a.errorbar(r[NOMBRES[t]], yy + (k - 0.5) * 0.25, xerr=np.abs(ic.T - r[NOMBRES[t]].to_numpy()),
                   fmt=mk, capsize=2, label=modelo)
    a.axvline(CNN[t], color="C3", ls="--", label="CNN (validación)")
    a.axvline(KATACHI[t], color="gray", ls=":", label="Katachi (prueba)")
    a.axvline(Y[t][va].std(), color="k", lw=0.6, label="σ de la etiqueta")
    a.set(xlabel=f"RMSE en validación · {NOMBRES[t]}")
ax[0].set_yticks(yy); ax[0].set_yticklabels(etiquetas, fontsize=8); ax[0].invert_yaxis()
ax[0].legend(fontsize=7, loc="lower right"); plt.tight_layout(); plt.show()

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

### 13.1 Residuos frente a la distancia

La red convolucional sobreestimaba la masa de las galaxias cercanas y subestimaba la de las lejanas. Revisamos el sesgo en masa por intervalo de redshift (los de la sección 7.2) para el modelo sin variables intrínsecas (S1) y para el modelo con las características seleccionadas (S4).

In [ ]:
fila_m = []
for nombre in ("S1 · aparente", f"S4 · seleccionadas ({K_SEL})"):
    res = PRED[(nombre, "HGB", "log_mstar")] - Y["log_mstar"][va]
    s = pd.Series(res).groupby(M.loc[va, "z_bin"].to_numpy()).median()
    fila_m.append(s.rename(nombre))
sesgo_z = pd.concat(fila_m, axis=1)
sesgo_z.index = [f"{bordes[i]:.3f}–{bordes[i + 1]:.3f}" for i in sesgo_z.index]
sesgo_z.index.name = "intervalo de z"
display(sesgo_z.round(3))

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

In [ ]:
# Productos para la siguiente fase (Model Engineering)
salida = df[["mangaid", "particion", *OBJETIVOS, *CONSTRUIDAS, "T_11"]]
guard.write(RESULTADOS / "caracteristicas_v1.parquet", lambda t: salida.to_parquet(t, index=False), allow_replace=True)
guard.write(RESULTADOS / "comparacion_validacion.csv", lambda t: RES.to_csv(t, index=False), allow_replace=True)
guard.write_json(RESULTADOS / "resumen_avance2.json", {
    "commit": COMMIT, "catalogo_sha256": SHA_CAT, "n_galaxias": int(len(df)), "n_construidas": len(CONSTRUIDAS),
    "seleccion": SELECCION, "n_pca": N_PCA, "n_factores": N_FA, "n_eigengalaxias": N_EIG}, allow_replace=True)
for p in sorted(RESULTADOS.glob("*.*")):
    print(f"  {p.name:40s} {p.stat().st_size / 1e6:7.2f} MB")

## 14. Conclusiones de la fase de preparación de los datos (CRISP-ML(Q))

En CRISP-ML(Q) (Studer et al. 2021), la fase de *Data Preparation* comprende cuatro tareas: selección de datos, limpieza, construcción de características y estandarización. Para cada tarea, la metodología pide identificar el **requisito** que se busca cumplir, el **riesgo** para la calidad del modelo y el **método de aseguramiento de calidad** que lo controla. La tabla resume cómo lo abordamos en este avance.

| Tarea | Requisito | Riesgo | Control de calidad aplicado |
|---|---|---|---|
| **Selección de datos** | Etiquetas confiables y particiones sin contaminación | Etiquetas erróneas; fuga de información | Solo QCFLAG = 0; partición del proyecto; galaxias de prueba sin usar; exclusión justificada de las variables derivadas de las etiquetas (sección 4) |
| **Limpieza** | Mediciones que reflejen la galaxia y no la observación | Cielo residual, vecinos, extinción, galaxias más grandes que el recorte, valores ausentes | Resta del cielo, enmascarado y relleno de vecinos, corrección SFD, indicador `apertura_truncada`, imputación con la mediana de entrenamiento (secciones 5 y 6) |
| **Construcción** | Variables con significado físico y medidas correctamente | Mediciones con errores de código o definiciones no estándar | Definiciones publicadas; 18 pruebas automáticas con galaxias sintéticas; validación contra NSA y el catálogo auxiliar (sección 5.5) |
| **Estandarización** | Variables comparables y bien condicionadas | Características sesgadas por asimetría; dominio de las de mayor escala; convergencia lenta; sesgo por telescopio y distancia | Regla de asimetría fijada de antemano; comparación de escaladores; prueba de convergencia; SMD y KS por región; pipeline reproducible ajustado solo con entrenamiento (secciones 8–10) |
| **Selección de características** | Menor complejidad sin perder información | Redundancia; sobreajuste por exceso de variables | Umbral de varianza, correlación, F, información mutua y χ²; número de variables elegido por validación cruzada con la regla de un error estándar (sección 11) |

> **Lectura de resultados:** *se completa con las salidas de la ejecución completa.*

**Qué pasa a la siguiente fase (*Model Engineering*):**

- `pipeline_preparacion.joblib`, que aplica las transformaciones sin reajustarlas.
- `caracteristicas_v1.parquet`, con todas las características medidas.
- `seleccion.json`, con el ranking y las características elegidas.
- `diccionario_caracteristicas.csv`, con la definición y referencia de cada una.

Con estos archivos podremos entrenar modelos tabulares y probar un **modelo híbrido** que combine la red convolucional con las características seleccionadas, en particular las intrínsecas, que aportan la información de distancia que la imagen por sí sola no contiene.

## 15. Referencias

**Metodología**

- Bulmer, M. G. (1979). *Principles of Statistics*. Dover.
- Austin, P. C. (2009). Balance diagnostics for comparing the distribution of baseline covariates between treatment groups in propensity-score matched samples. *Statistics in Medicine*, 28, 3083–3107.
- Box, G. E. P. & Cox, D. R. (1964). An analysis of transformations. *Journal of the Royal Statistical Society B*, 26, 211–252.
- Galli, S. (2022). *Python Feature Engineering Cookbook* (2.ª ed.). Packt.
- Hastie, T., Tibshirani, R. & Friedman, J. (2009). *The Elements of Statistical Learning* (2.ª ed.). Springer.
- Horn, J. L. (1965). A rationale and test for the number of factors in factor analysis. *Psychometrika*, 30, 179–185.
- Kaiser, H. F. (1974). An index of factorial simplicity. *Psychometrika*, 39, 31–36.
- Studer, S., Bui, T. B., Drescher, C., et al. (2021). Towards CRISP-ML(Q): A machine learning process model with quality assurance methodology. *Machine Learning and Knowledge Extraction*, 3, 392–413.
- Yeo, I.-K. & Johnson, R. A. (2000). A new family of power transformations to improve normality or symmetry. *Biometrika*, 87, 954–959.

**Astronomía**

- Alfonzo, J. P., Iyer, K. G., Akiyama, M., et al. (2024). Decoding the imprints of past star formation on present-day morphology in galaxies with interpretable CNNs. *The Astrophysical Journal*, 967, 152.
- Bell, E. F., McIntosh, D. H., Katz, N. & Weinberg, M. D. (2003). The optical and near-infrared properties of galaxies. I. *The Astrophysical Journal Supplement Series*, 149, 289.
- Blanton, M. R., Dalcanton, J., Eisenstein, D., et al. (2001). The luminosity function of galaxies in SDSS commissioning data. *The Astronomical Journal*, 121, 2358.
- Conselice, C. J. (2003). The relationship between stellar light distributions of galaxies and their formation histories. *The Astrophysical Journal Supplement Series*, 147, 1.
- Dey, A., Schlegel, D. J., Lang, D., et al. (2019). Overview of the DESI Legacy Imaging Surveys. *The Astronomical Journal*, 157, 168.
- Ebrová, I., Bílek, M. & Eliášek, J. (2025). Photometric stellar masses for galaxies in DESI Legacy Imaging Surveys. *Astronomy & Astrophysics*, 704, A232.
- Lotz, J. M., Primack, J. & Madau, P. (2004). A new nonparametric approach to galaxy morphological classification. *The Astronomical Journal*, 128, 163.
- Lupton, R., Blanton, M. R., Fekete, G., et al. (2004). Preparing red-green-blue images from CCD data. *Publications of the Astronomical Society of the Pacific*, 116, 133.
- Rodriguez-Gomez, V., Snyder, G. F., Lotz, J. M., et al. (2019). The optical morphologies of galaxies in the IllustrisTNG simulation. *Monthly Notices of the Royal Astronomical Society*, 483, 4140.
- Sánchez, S. F., Barrera-Ballesteros, J. K., Lacerda, E., et al. (2022). SDSS-IV MaNGA: pyPipe3D analysis release for 10,000 galaxies. *The Astrophysical Journal Supplement Series*, 262, 36.
- Schlafly, E. F. & Finkbeiner, D. P. (2011). Measuring reddening with Sloan Digital Sky Survey stellar spectra and recalibrating SFD. *The Astrophysical Journal*, 737, 103.
- Schlegel, D. J., Finkbeiner, D. P. & Davis, M. (1998). Maps of dust infrared emission for use in estimation of reddening and cosmic microwave background radiation foregrounds. *The Astrophysical Journal*, 500, 525.
- Strateva, I., Ivezić, Ž., Knapp, G. R., et al. (2001). Color separation of galaxy types in the Sloan Digital Sky Survey imaging data. *The Astronomical Journal*, 122, 1861.
- Uzeirbegovic, E., Geach, J. E. & Kaviraj, S. (2020). Eigengalaxies: describing galaxy morphology using principal components in image space. *Monthly Notices of the Royal Astronomical Society*, 498, 4021.
- Walmsley, M., Géron, T., Kruk, S., et al. (2023). Galaxy Zoo DESI: Detailed morphology measurements for 8.7M galaxies in the DESI Legacy Imaging Surveys. *Monthly Notices of the Royal Astronomical Society*, 526, 4768.
- Willmer, C. N. A. (2018). The absolute magnitude of the Sun in several filters. *The Astrophysical Journal Supplement Series*, 236, 47.
- Ye, R., Shen, S., de Souza, R. S., et al. (2025). From Galaxy Zoo DECaLS to BASS/MzLS: detailed galaxy morphology classification with unsupervised domain adaption. *Monthly Notices of the Royal Astronomical Society*, 537, 640.